# Super Resolution SRCNN Pipeline for Remote Run (Streaming)
This notebook contains the complete pipeline to run Model Training sequentially on a remote machine like Colab. 
It uses Hugging Face `datasets` in **streaming mode** to dynamically load and process data on the fly, entirely avoiding massive dataset downloads and saving significant disk space and processing time.

In [ ]:
import sys
!{sys.executable} -m pip install datasets tqdm opencv-python h5py tensorboard lpips scikit-image pyyaml

In [ ]:
import os
import time
from pathlib import Path
import cv2
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, IterableDataset
from torch.utils.tensorboard import SummaryWriter
from skimage.metrics import peak_signal_noise_ratio as psnr_func
from skimage.metrics import structural_similarity as ssim_func
from datasets import load_dataset
from tqdm import tqdm
import lpips
import logging

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s: %(message)s")
logger = logging.getLogger("remote_training")

In [ ]:
from dataclasses import dataclass

@dataclass
class StreamingTrainingConfig:
    root_dir: Path
    model_path: Path
    model_type: str
    hf_dataset_name: str
    hf_dataset_config: str
    patch_size: int
    stride: int
    epochs: int
    batch_size: int
    learning_rate: float
    normalization: str
    device: str
    patience: int
    log_step: int

# Configuration Instance
training_config = StreamingTrainingConfig(
    root_dir=Path("artifacts/model_training"),
    model_path=Path("artifacts/model_training/srcnn.pth"),
    model_type="srcnn",
    hf_dataset_name="eugenesiow/Div2k",
    hf_dataset_config="bicubic_x4",
    patch_size=96,
    stride=96,
    epochs=100,
    batch_size=16,
    learning_rate=0.0001,
    normalization="zero_to_one",
    device="cuda",
    patience=7,
    log_step=100
)

# Create mandatory directory immediately
os.makedirs(training_config.root_dir, exist_ok=True)
os.makedirs(training_config.root_dir / "logs", exist_ok=True)

In [ ]:
class SRCNN(nn.Module):
    def __init__(self, num_channels=3):
        super(SRCNN, self).__init__()
        self.conv1 = nn.Conv2d(num_channels, 64, kernel_size=9, padding=9 // 2)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(64, 32, kernel_size=5, padding=5 // 2)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv3 = nn.Conv2d(32, num_channels, kernel_size=5, padding=5 // 2)

    def forward(self, x):
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        return self.conv3(x)

class StreamingSRCNNDataset(IterableDataset):
    def __init__(self, hf_dataset_stream, patch_size=96, stride=96, normalization='zero_to_one', shuffle_buffer=1000):
        super().__init__()
        self.hf_dataset = hf_dataset_stream
        self.patch_size = patch_size
        self.stride = stride
        self.normalization = normalization
        self.shuffle_buffer = shuffle_buffer

    def __iter__(self):
        buffer = []
        for item in self.hf_dataset:
            # Extract PIL Images and convert to NumPy arrays
            # eugenesiow/Div2k usually has 'hr' and 'lr' keys
            hr_img = np.array(item['hr'].convert('RGB'))
            lr_img = np.array(item['lr'].convert('RGB'))
            
            if hr_img is None or lr_img is None: continue
            
            # Upscale LR to HR dimension via bicubic interpolation as per SRCNN paper preprocessing
            lr_upscaled = cv2.resize(lr_img, (hr_img.shape[1], hr_img.shape[0]), interpolation=cv2.INTER_CUBIC)
            
            # Extract dense patches
            current_patches_hr, current_patches_lr = [], []
            for y in range(0, hr_img.shape[0] - self.patch_size + 1, self.stride):
                for x in range(0, hr_img.shape[1] - self.patch_size + 1, self.stride):
                    current_patches_hr.append(hr_img[y:y+self.patch_size, x:x+self.patch_size])
                    current_patches_lr.append(lr_upscaled[y:y+self.patch_size, x:x+self.patch_size])
            
            # Yield buffered items in random order
            for hr_patch, lr_patch in zip(current_patches_hr, current_patches_lr):
                hr_patch = hr_patch.astype(np.float32)
                lr_patch = lr_patch.astype(np.float32)

                if self.normalization == "zero_to_one":
                    hr_patch /= 255.0; lr_patch /= 255.0
                else:
                    hr_patch = (hr_patch / 127.5) - 1.0; lr_patch = (lr_patch / 127.5) - 1.0
                
                # HWC to CHW
                hr_tensor = torch.from_numpy(hr_patch).permute(2, 0, 1)
                lr_tensor = torch.from_numpy(lr_patch).permute(2, 0, 1)

                buffer.append((lr_tensor, hr_tensor))
                
                if len(buffer) >= self.shuffle_buffer:
                    idx = np.random.randint(0, len(buffer))
                    yield buffer.pop(idx)
                    
        # Flush remaining buffer and shuffle slightly
        np.random.shuffle(buffer)
        for p in buffer:
            yield p

def calculate_psnr(img1, img2):
    return psnr_func(img1, img2, data_range=img2.max() - img2.min())

class EarlyStopping:
    def __init__(self, patience=7, min_delta=0):
        self.patience = patience; self.min_delta = min_delta; self.counter = 0; self.best_loss = None; self.early_stop = False
    def __call__(self, val_loss):
        if self.best_loss is None: self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
        else: self.best_loss = val_loss; self.counter = 0

In [ ]:
class ModelTraining:
    def __init__(self, config: StreamingTrainingConfig):
        self.config = config
        self.device = torch.device(config.device if torch.cuda.is_available() else "cpu")
        log_dir = self.config.root_dir / "logs"
        self.writer = SummaryWriter(log_dir=str(log_dir))
        if torch.cuda.is_available():
            self.scaler = torch.amp.GradScaler('cuda')
        else:
            self.scaler = None

    def train(self):
        # The eugenesiow/Div2k validation split is typically 'validation'
        train_loader = self._get_dataloader('train')
        valid_loader = self._get_dataloader('validation')

        model = SRCNN().to(self.device)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=self.config.learning_rate)
        
        early_stopping = EarlyStopping(patience=self.config.patience)
        best_psnr = 0.0; global_step = 0

        if os.path.exists(self.config.model_path):
            logger.info(f"Loading weights from {self.config.model_path}")
            model.load_state_dict(torch.load(self.config.model_path, map_location=self.device))

        for epoch in range(self.config.epochs):
            model.train()
            epoch_loss = 0.0
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{self.config.epochs}")
            
            batch_idx = 0
            for batch_idx, (lr, hr) in enumerate(progress_bar):
                lr, hr = lr.to(self.device), hr.to(self.device)
                optimizer.zero_grad()
                
                if self.scaler:
                    with torch.amp.autocast('cuda'):
                        outputs = model(lr)
                        loss = criterion(outputs, hr)
                    self.scaler.scale(loss).backward()
                    self.scaler.step(optimizer)
                    self.scaler.update()
                else:
                    outputs = model(lr)
                    loss = criterion(outputs, hr)
                    loss.backward()
                    optimizer.step()

                epoch_loss += loss.item()
                global_step += 1

                if global_step % self.config.log_step == 0:
                    avg_step_loss = epoch_loss / (batch_idx + 1)
                    self.writer.add_scalar("Loss/Train_Step", avg_step_loss, global_step)
                    progress_bar.set_postfix({"loss": f"{avg_step_loss:.5f}"})

            # After exhausting iterable for this epoch
            avg_val_loss, avg_psnr = self._validate(model, valid_loader, criterion)
            self.writer.add_scalar("Loss/Validation", avg_val_loss, epoch)
            self.writer.add_scalar("Metrics/PSNR_dB", avg_psnr, epoch)
            logger.info(f"Epoch {epoch+1}: Val Loss: {avg_val_loss:.6f} | Val PSNR: {avg_psnr:.2f}dB")

            if avg_psnr > best_psnr:
                best_psnr = avg_psnr
                torch.save(model.state_dict(), self.config.model_path)
                logger.info(f"New Best Model saved at {best_psnr:.2f}dB PSNR")

            early_stopping(avg_val_loss)
            if early_stopping.early_stop:
                logger.warning(f"Early Stopping triggered at epoch {epoch+1}")
                break

            # Because Streaming Datasets exhaust themselves, it's safer to re-create loaders
            # But Hugging Face IteratorDataset automatically restarts iteration upon 'for x in ...'
            
        self.writer.close()
        logger.info("Training process finalized.")

    def _validate(self, model, loader, criterion):
        model.eval(); val_loss = 0.0; psnr_values = []
        logger.info("Running Validation...")
        with torch.no_grad():
            val_steps = 0
            for lr, hr in tqdm(loader, desc="Validation"):
                lr, hr = lr.to(self.device), hr.to(self.device)
                if self.scaler:
                    with torch.amp.autocast('cuda'):
                        outputs = model(lr)
                        loss = criterion(outputs, hr)
                else:
                    outputs = model(lr)
                    loss = criterion(outputs, hr)
                    
                val_loss += loss.item()
                out_np = outputs.cpu().numpy() * 255.0
                hr_np = hr.cpu().numpy() * 255.0
                psnr_values.append(calculate_psnr(out_np, hr_np))
                val_steps += 1
        return val_loss / max(1, val_steps), np.mean(psnr_values)

    def _get_dataloader(self, split):
        logger.info(f"Loading HF dataset split: {split} (streaming)")
        # Using streaming avoids full download
        hf_ds = load_dataset(self.config.hf_dataset_name, name=self.config.hf_dataset_config, split=split, streaming=True)
        dataset = StreamingSRCNNDataset(
            hf_ds,
            patch_size=self.config.patch_size,
            stride=self.config.stride,
            normalization=self.config.normalization,
            # Buffer for random patch shuffling during training
            shuffle_buffer=2000 if split == 'train' else 10
        )
        return DataLoader(
            dataset, batch_size=self.config.batch_size, shuffle=False,
            pin_memory=True, num_workers=0
        )

In [ ]:
# Execute Training
# To visualize in tensorboard in Colab, run this command in a separate cell before training:
# %load_ext tensorboard
# %tensorboard --logdir artifacts/model_training/logs

logger.info("--- Starting Model Training (Streaming Mode) ---")
mt = ModelTraining(training_config)
mt.train()